In [1]:
#Loader
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
Loadeee1=PyPDFLoader("Reading4-NLP.pdf")
Loadeee2=PyPDFLoader("Tutorial_EDIT.pdf")
Loadeee3=PyPDFLoader("Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf")
docs1=Loadeee1.load()
docs2=Loadeee2.load()
docs3=Loadeee3.load()
docss=docs1+docs2+docs3
print(len(docss))
print (docs1[33].metadata)
print(docs2[2].page_content[:7])

C:\Users\waadt\AppData\Local\Temp\ipykernel_39876\1472226516.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 194 0 (offset 0)


611
{'producer': 'macOS Version 12.1 (Build 21C52) Quartz PDFContext', 'creator': 'Safari', 'creationdate': "D:20220116035409Z00'00'", 'title': 'Natural Language Processing (NLP) with Python\u200a—\u200aTutorial\xa0 – Towards AI — The World’s Leading AI and Technology Publication', 'author': 'Li, Peilong', 'moddate': "D:20220116035409Z00'00'", 'source': 'Reading4-NLP.pdf', 'total_pages': 64, 'page': 33, 'page_label': '34'}
CONTENT


In [2]:
import numpy as np 
from sentence_transformers import SentenceTransformer
from typing import List

In [4]:
class EmbeddingBoss:
        def __init__(self,model_name :str = "all-MiniLM-L6-v2"):
                self.model_name=model_name
                self.model=None
                self._load_model()
        def  _load_model(self):
            try:
                print(f"loading model is done")
                self.model=SentenceTransformer(self.model_name)
            except Exception as e:
                print(f"model failed")
                raise

        def generate_embedding(self,textt:List[str]) ->np.ndarray:

            if not self.model:
                raise ValueError("Model not Loaded")
            print(f"number of str is{len(textt)}")
            embeddings= self.model.encode(
            textt,
            show_progress_bar=True)
            print(f"shape of generatin embedding is {embeddings.shape}")
            return embeddings

        def get_embedding_dimension(self) -> int:
            if not self.model:
                raise ValueError("model not loaded")
            return self.model.get_sentence_embedding_dimension()
embeddingss= EmbeddingBoss()
embeddingss









loading model is done


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
#chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
Splitters=RecursiveCharacterTextSplitter()
chunk_size=2000
chunk_overlap=150
seperator=["\n\n","\n",".","",""]
chunks=Splitters.split_documents(docss)
print(f"pages: {len(docss)}-> chunks:{len(chunks)}")
textt=[c.page_content for c in chunks]
embedding= embeddingss.generate_embedding(textt)
print(embedding.shape)














pages: 611-> chunks:611
number of str is611


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

shape of generatin embedding is (611, 384)
(611, 384)


In [22]:
embedding = embeddingss.generate_embedding(textt)
import chromadb
client = chromadb.PersistentClient(path=r"C:\Users\waadt\Desktop\RAG2\chroma_db")
collection = client.get_or_create_collection("course_docs",
    metadata={"hnsw:space":"cosine"}
)
collection.add(

    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=textt,
    embeddings=embedding.tolist(),
    metadatas=[
        {k: v for k, v in c.metadata.items()
         if isinstance(v, (str, int, float, bool))}
        for c in chunks
    ]
)
print(len(chunks), len(textt), embedding.shape)


c = chromadb.PersistentClient(path=r"C:\Users\waadt\Desktop\RAG2\chroma_db")
print([(col.name, col.count()) for col in c.list_collections()])

number of str is611


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

shape of generatin embedding is (611, 384)
611 611 (611, 384)
[('pdf_chunks', 611), ('docs', 0), ('course_docs', 611)]


In [16]:
query="What is NLP?"
q_vec=embeddingss.generate_embedding([query])
results=collection.query(query_embeddings=q_vec.tolist(), n_results=8)
for doc, meta, dist in zip(results["documents"][0],results['metadatas'][0],results['distances'][0]):
    print(f"distance:{dist:.3f}| page:{meta.get('page')}")
    print(doc[:300],"\n---")




number of str is1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

shape of generatin embedding is (1, 384)
distance:0.397| page:1
10. Stemming
11. Lemmatization
12. Part-of-Speech (PoS) tagging
13. Chunking
14. Chinking
15. Named Entity Recognition (NER)
16. WordNet
17. Bag of Words
18. TF-IDF
What is Natural Language Processing?
Computers and machines are great at working with tabular data or
spreadsheets. However, as human b 
---
distance:0.448| page:4
take much time, and it requires manual effort.
Statistical Natural Language Processing:
It uses large amounts of data and tries to derive conclusions from it.
Statistical NLP uses machine learning algorithms to train NLP models. After
successful training on large amounts of data, the trained model w 
---
distance:0.455| page:61
Figure 138: Final TF-IDF values.
Python Implementation:
Figure 139: Python implementation of TF-IDF code snippet.
Figure 140: Final output.
Conclusion:
These are some of the basics for the exciting field of natural language
processing (NLP). We hope you enjoyed reading this ar

In [18]:
context = "\n\n".join(results["documents"][0])

prompt = f"""Answer the question based on the context below.
The answer is present in the context, so find it carefully.
Do not say "I don't know" if the context contains the answer.

Context:
{context}

Question: {query}
Answer:"""

In [ ]:
from google import genai

client = genai.Client(api_key="put your api key")

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)
print(response.text)

Based on the context provided, Natural Language Processing (NLP) is a subfield of artificial intelligence, in which its depth involves the interactions between computers and humans, with the goal of making computers understand unstructured text and retrieve meaningful pieces of information from it.
